### Day 5 Assignment: PySpark DataFrame Transformations & Git Integration

### Basic Tasks

#### 1. Load e-commerce dataset

In [0]:
df = spark.read.csv("/Volumes/dev/demo/ex-volume/messy_ecommerce_sales.csv", header=True, inferSchema=True)
df.display()

In [0]:
from pyspark.sql.functions import col
# 1. Rows containing NULLs
df.filter(
    col("sale_id").isNull() |
    col("customer_id").isNull() |
    col("product_id").isNull() |
    col("quantity").isNull() |
    col("sale_amount").isNull() |
    col("sale_date").isNull() |
    col("region").isNull()
).show()

In [0]:
# 2. Compare total and distinct rows
print("Total rows:", df.count())
print("Distinct rows:", df.distinct().count())

In [0]:
# 3. Remove duplicate rows
df_clean = df.dropDuplicates()
print("Rows after removing duplicates:", df_clean.count())

#### 2. Rename columns

In [0]:
df_renamed_cols = df_clean.withColumnRenamed("sale_id", "sale_ID")\
.withColumnRenamed("customer_id", "customer_ID")\
.withColumnRenamed("product_id", "product_ID")
df_renamed_cols.display()

#### 3. Connect Databricks Repo to Git 


Created the initial e-commerce data cleaning notebook, committed the changes, and pushed them to the Git repository with the commit message: "Add initial e-commerce data cleaning notebook".

###  Intermediate Tasks

####4. Build a cleaning pipeline

In [0]:
df_cleaned = df_renamed_cols.dropna(how="all").dropDuplicates().fillna({"quantity": 0, "sale_amount": 0, "region": "Unknown"}).sort("sale_date")
df_cleaned.display()

#### 5. Aggregation and JOIN

In [0]:
#Sample region dataset
region_data = [
    ("North", "Northern Region"),
    ("South", "Southern Region"),
    ("East", "Eastern Region"),
    ("West", "Western Region"),
    ("Unknown", "Unknown Region")
]

region_df = spark.createDataFrame(
    region_data,
    ["region", "region_name"]
)

display(region_df)

In [0]:
revenue_by_region = df_cleaned.groupBy("region")\
    .agg({"sale_amount": "sum"})\
    .withColumnRenamed("sum(sale_amount)", "total_revenue")\
    .join(region_df, on ="region", how = "left")
revenue_by_region.display()

#### 6. Create a feature branch

![image_1787648121179.png](./image_1787648121179.png "image_1787648121179.png")
Created the `feature/improve-cleaning-logic`
 branch and committed the updated cleaning logic.

Created a pull request, which was successfully merged into the main branch and closed.

### Advanced Tasks 

#### 7. Extend the pipeline

In [0]:
messy_df = spark.read.csv("/Volumes/dev/demo/ex-volume/messy_ecommerce_advanced_task7.csv", header=True, inferSchema=True)
messy_df.display()

In [0]:
from pyspark.sql.functions import *
clean_df = (
    messy_df.na.drop(how="all")
    
    # Remove currency symbols and commas
    .withColumn(
        "sale_amount",
         regexp_replace(col("sale_amount"), r"[$₹€£,]", "").cast("double")
    )
    
    .withColumn(
    "sale_date",
    coalesce(
        try_to_date(col("sale_date"), "yyyy-MM-dd"),
        try_to_date(col("sale_date"), "dd/MM/yyyy"),
        try_to_date(col("sale_date"), "dd-MM-yyyy"),
        try_to_date(col("sale_date"), "MMM dd, yyyy"),
        try_to_date(col("sale_date"), "yyyy/MM/dd"),
        try_to_date(col("sale_date"), "dd.MM.yyyy"),
        try_to_date(col("sale_date"), "MMMM d yyyy"),
        try_to_date(col("sale_date"), "dd-MM-yy"),
        try_to_date(col("sale_date"), "dd-MMM-yyyy")
    )
    )
    .fillna({
        "customer_id": 0,
        "product_id": 0,
        "quantity": 0,
        "sale_amount": 0.0,
        "region": "Unknown"
    })
    
    .dropDuplicates()
    .orderBy("sale_date")
)

display(clean_df)

Dirty Data Handling

The 'sale_amount'[](url) column contained currency symbols and comma separators, which prevented reliable numeric calculations. Thus, removed the currency symbols and commas using `regexp_replace()` and converted the cleaned values to 'double'.

Then handled remaining missing values using sensible defaults, removed exact duplicates, and sorted the data by sale date.

#### 8. Dev/Main branching strategy


Git Branching & Pull Request Workflow

- 'main' contains reviewed and production-ready code.
- Create a separate feature branch from 'main' for every new change.
- Use descriptive branch names such as `feature/data-cleaning`, `feature/sales-analysis`, `feature/data-quality`.
- Commit changes with clear and meaningful commit messages.
- Push the feature branch and open a Pull Request against 'main'.
- At least one teammate should review the changes before merging.
- Address review comments and push any required updates to the same branch.
- Merge the Pull Request only after approval and successful checks.
- Delete the feature branch after the Pull Request is merged.

This workflow keeps 'main' stable and makes changes easier to review, track, and revert.

#### 9. Business summary report

In [0]:
revenue_by_region = clean_df\
    .groupBy("region")\
    .agg(sum("sale_amount")\
    .alias("total_revenue"))\
    .orderBy(desc("total_revenue"))

display(revenue_by_region)

# This tells the business:
# Which region is contributing the most revenue?

In [0]:
avg_order_value = (
    clean_df
    .groupBy("region")
    .agg(
        avg("sale_amount").alias("average_order_value")
    )
    .orderBy(desc("average_order_value"))
)

display(avg_order_value)

# This helps answer:
# Which region has the highest-value orders on average?

In [0]:
monthly_revenue = (
    clean_df
    .withColumn("month", date_format("sale_date", "yyyy-MM"))
    .groupBy("month")
    .agg(sum("sale_amount").alias("monthly_revenue"))
    .orderBy("month")
)

display(monthly_revenue)

# This tells How does revenue change month over month?

#### Business & Data-Quality Caveats

- Missing customer or product IDs were replaced with "0", so some transactions cannot be reliably attributed to a specific customer or product.
- Missing regions were classified as "Unknown", which may slightly affect regional comparisons.
- Missing or invalid sales amounts may affect revenue and average-order-value calculations.
- Duplicate records were removed based on exact row matches; non-identical duplicate transactions may still exist.